In [50]:
from z3 import *

In [51]:
flag_input = [BitVec(f'FLAG_{i}', 8) for i in range(22)]
flag_input

[FLAG_0,
 FLAG_1,
 FLAG_2,
 FLAG_3,
 FLAG_4,
 FLAG_5,
 FLAG_6,
 FLAG_7,
 FLAG_8,
 FLAG_9,
 FLAG_10,
 FLAG_11,
 FLAG_12,
 FLAG_13,
 FLAG_14,
 FLAG_15,
 FLAG_16,
 FLAG_17,
 FLAG_18,
 FLAG_19,
 FLAG_20,
 FLAG_21]

In [52]:
static_bit_map_processing_data = [
    0xaa, 0x6a, 0xaa, 0x9a,
    0x9a, 0x96, 0xa8, 0xaa,
    0xaa, 0x2a, 0xa6, 0xaa,
    0x0a, 0x6a, 0xaa, 0x88,
    0x9a, 0x2a, 0xa8, 0xa8,
    0xaa, 0xaa, 0xaa, 0x2a,
    0x88, 0xa6, 0xa6, 0xaa,
    0x66, 0x68, 0xaa, 0xa8,
    0x8a, 0xaa, 0xa6, 0x6a,
    0xaa, 0xaa, 0xa8, 0x28,
    0xaa, 0xaa, 0xa6, 0xa9,
    0xa2, 0x8a, 0x28, 0xaa,
    0x12, 0x00, 0x00, 0x00
]

In [53]:
def xor_array(arr1, arr2):
    return [a ^ b for a, b in zip(arr1, arr2)]

step_1_static_otp = bytes.fromhex('6453332124355762073b231954040f497463541f3f54')
step_1_static_otp = [BitVecVal(b, 8) for b in step_1_static_otp]
step_1_result = xor_array(flag_input, step_1_static_otp)
step_1_result += [BitVecVal(0, 8)]

In [54]:

step_2_result = [0] * 0xC4

step_1_result_index = 0
static_bit_map_processing_data_index = 0
computational_memory_index = 0

current_result_part = step_1_result[step_1_result_index]
current_comparator = static_bit_map_processing_data[static_bit_map_processing_data_index]

result_part_ctr = 0
comparator_ctr = 0

while True:
    test = current_comparator & 0x03

    if test == 0x02:
        test = current_result_part
        current_result_part = LShR(current_result_part, 1)
        result_part_ctr += 1

        if result_part_ctr == 0x07:
            step_1_result_index += 1
            current_result_part = step_1_result[step_1_result_index]
            result_part_ctr = 0
    else:
        test = BitVecVal(test, 8)

    test &= 1
    step_2_result[computational_memory_index] = test
    computational_memory_index += 1

    if computational_memory_index == len(step_2_result):
        break

    current_comparator >>= 2
    comparator_ctr += 1

    if comparator_ctr != 0x04:
        continue

    static_bit_map_processing_data_index += 1
    current_comparator = static_bit_map_processing_data[static_bit_map_processing_data_index]
    comparator_ctr = 0

step_2_result = list(map(simplify, step_2_result))

In [55]:
def mystery_1_translated(inner_movement, outer_movement):
    computational_memory_index = 0

    for _ in range(0x0E):
        inner_computational_index = computational_memory_index

        last_known = 0xFF
        same_seq_ctr = 0
        zero_ctr = 0
        total_ctr = 0

        while True:
            currently_looked_at = step_2_result[inner_computational_index]
            inner_computational_index += inner_movement

            if currently_looked_at != last_known:
                same_seq_ctr = 0
                last_known = currently_looked_at

            same_seq_ctr += 1

            if same_seq_ctr == 0x03:
                return 0
            
            if currently_looked_at == 0x00:
                zero_ctr += 1

            total_ctr += 1

            if total_ctr == 0x0E:
                if zero_ctr != 0x07:
                    return 0
                
                break

        computational_memory_index += outer_movement

    return 1

In [56]:
myster_2_compute_memory = [0] * 0xFF

def mystery_2_translated(inner_movement, outer_movement):
    computational_memory_index = 0
    extra_index = 0

    for _ in range(0x0E):
        inner_computational_index = computational_memory_index
        calculated_field = 0

        for _ in range(0x0E):
            calculated_field <<= 1
            calculated_field |= step_2_result[inner_computational_index]
            inner_computational_index += inner_movement

        for extra_field in range(extra_index):
            if myster_2_compute_memory[extra_field] == calculated_field:
                return 0

        myster_2_compute_memory[extra_index] = calculated_field
        extra_index += 1
        computational_memory_index += outer_movement

    return 1

In [57]:
def mystery_1_z3(inner_movement, outer_movement):
    requirements = []
    computational_memory_index = 0

    for _ in range(0x0E):
        inner_computational_index = computational_memory_index
        last_3 = []
        total_sum = BitVecVal(0, 8)

        for _ in range(0x0E):
            currently_looked_at = step_2_result[inner_computational_index]
            inner_computational_index += inner_movement

            # Requirement not 3 in a row
            last_3.append(currently_looked_at)

            if len(last_3) == 0x03:
                last_3_sum = simplify(last_3[0] + last_3[1] + last_3[2])
                last_3.pop(0)
                requirements.append(last_3_sum != 0x03)
                requirements.append(last_3_sum != 0x00)
            
            total_sum += currently_looked_at
            
        # Requirement 7 zeros and 7 ones
        requirements.append(simplify(total_sum) == 0x07)
        computational_memory_index += outer_movement

    return requirements

In [58]:
myster_2_compute_memory = [0] * 0xFF

def mystery_2_z3(inner_movement, outer_movement):
    requirements = []
    computational_memory_index = 0
    extra_index = 0

    for _ in range(0x0E):
        inner_computational_index = computational_memory_index
        calculated_field = BitVecVal(0, 16)

        for _ in range(0x0E):
            calculated_field <<= 1
            calculated_field |= Concat(BitVecVal(0, 8), step_2_result[inner_computational_index])
            inner_computational_index += inner_movement

        calculated_field = simplify(calculated_field)

        for extra_field in range(extra_index):
            requirements.append(myster_2_compute_memory[extra_field] != calculated_field)

        myster_2_compute_memory[extra_index] = calculated_field
        extra_index += 1
        computational_memory_index += outer_movement

    return requirements

In [59]:
def all_flag_high_bit_zero():
    requirements = []
    for flag_part in flag_input:
        requirements.append(simplify(Extract(7, 7, flag_part)) == BitVecVal(0, 1))
    return requirements

In [60]:
_requirements = []
_requirements.extend(all_flag_high_bit_zero())
_requirements.extend(mystery_1_z3(0x01, 0x0E))
_requirements.extend(mystery_1_z3(0x0E, 0x01))
_requirements.extend(mystery_2_z3(0x01, 0x0E))
_requirements.extend(mystery_2_z3(0x0E, 0x01))

In [61]:
solver = Solver()

for requirement in _requirements:
    solver.add(requirement)

In [62]:
solver.check()

sat

In [63]:
for i in range(22):
    print(f'{chr(solver.model()[flag_input[i]].as_long())}', end='')

AuRuM_fulmiN@N$_a825ec